# 261. Graph Valid Tree
**Difficulty:** 🟡 Medium (Premium) · **Topic:** Graph · **LeetCode:** https://leetcode.com/problems/graph-valid-tree/

## 💡 Concepts

**Core concept(s):** A graph is a tree iff it is **fully connected** and has **no cycle** — checkable with **Union-Find** or one traversal.

**Why it applies here:** A tree on `n` nodes has exactly `n-1` edges, is all in one piece, and has no loops. If edges != n-1 it can't be a tree. Otherwise, Union-Find merging edges: if two endpoints are already in the same group, that edge makes a cycle → not a tree.

**Key intuition:** Exactly n-1 edges + everything connected + no cycle = a tree.

---

### 📚 What is a Graph?
A **graph** is dots (**nodes/vertices**) joined by lines (**edges**). Edges can be **directed** (one-way, like prerequisites) or **undirected** (two-way, like friendships). A **grid** is just a graph where each cell links to its neighbors.
- **In Python:** usually an **adjacency list** — a `dict` mapping each node to the list of nodes it connects to.

### 📚 What is Union-Find (Disjoint Set)?
**Union-Find** tracks which items are in the same group. `find(x)` returns x's group leader; `union(a,b)` merges two groups. With path-compression it's almost **O(1)** per call.
- **Great for:** counting connected pieces, detecting cycles in undirected graphs.
- **In Python:** a `parent` array where each node points toward its group's leader.

### 📚 What is DFS (Depth-First Search)?
**DFS** follows one path as deep as it goes, then backtracks. On graphs you must remember **visited** nodes so you don't loop forever.
- **Complexity:** **O(V + E)** — each node and edge once.
- **In Python:** recursion or an explicit stack, plus a `visited` set.

---

**Prerequisite knowledge:**
- Union-Find, or a connectivity traversal.
- The edge-count fact for trees.

## 📝 Problem

Given `n` nodes (0..n-1) and a list of undirected `edges`, return `True` if they form a valid tree.

**Example**
```
5, [[0,1],[0,2],[0,3],[1,4]] -> True
5, [[0,1],[1,2],[2,3],[1,3],[1,4]] -> False   (has a cycle)
```

> Two approaches, both `O(n)`-ish: Union-Find and a connectivity traversal.

### Approach 1 — Union-Find

**Idea:** A tree needs exactly `n-1` edges. Merge each edge; if its two ends are already connected, that's a cycle → not a tree.

**Time:** `O(n · α) ≈ O(n)`. **Space:** `O(n)`.

In [ ]:
def valid_tree_uf(n, edges):
    if len(edges) != n - 1:                # a tree on n nodes has EXACTLY n-1 edges
        return False
    parent = list(range(n))                # each node starts as its own group leader
    def find(x):                           # find x's group leader (with path compression)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra == rb:                       # both ends already connected -> this edge makes a cycle
            return False
        parent[ra] = rb                    # otherwise merge the two groups
    return True                            # n-1 edges + no cycle => connected tree

### Approach 2 — Connectivity Traversal

**Idea:** With `n-1` edges, the only thing left to check is that everything is in one piece. DFS/BFS from node 0 and confirm you reach all `n` nodes.

**Time:** `O(n)`. **Space:** `O(n)`.

In [ ]:
from collections import defaultdict

def valid_tree_dfs(n, edges):
    if len(edges) != n - 1:                # wrong edge count can't be a tree
        return False
    graph = defaultdict(list)              # build an undirected adjacency list
    for a, b in edges:
        graph[a].append(b); graph[b].append(a)
    seen = set(); stack = [0]              # explore from node 0
    while stack:
        node = stack.pop()
        if node in seen:
            continue
        seen.add(node)
        for nb in graph[node]:
            if nb not in seen:
                stack.append(nb)
    return len(seen) == n                  # reached every node -> fully connected

In [ ]:
# Correctness check
tests = [
    (5, [[0,1],[0,2],[0,3],[1,4]], True),
    (5, [[0,1],[1,2],[2,3],[1,3],[1,4]], False),
    (4, [[0,1],[2,3]], False),                # right edge count? no (2 != 3) -> False
    (1, [], True),
]
for n, edges, exp in tests:
    a, b = valid_tree_uf(n, edges), valid_tree_dfs(n, edges)
    print(f"n={n}, edges={edges} -> uf={a}, dfs={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)` / `O(V+E)` | ≈ **2×** |
| `O(n log n)`      | ≈ **2×** (slightly more) |
| `O(n²)`           | ≈ **4×** |

Inputs are shaped to force the worst case while keeping recursion shallow (stars / checkerboards) so nothing overflows the stack.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    edges = [[0, i] for i in range(1, n)]   # a star: valid tree, n-1 edges
    return (n, edges)
solutions = {
    "union-find O(n)": valid_tree_uf,
    "traversal  O(n)": valid_tree_dfs,
}
sizes = [2000, 4000, 8000, 16000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Tree = connected + acyclic + n-1 edges:** checking the edge count first is a cheap early filter.
- **Union-Find detects cycles:** merging an edge whose ends are already joined reveals a loop.
- **Signal:** "is this a tree", "connected and no cycle".
- **Related problems:** Number of Connected Components, Redundant Connection, Number of Islands.
- **Common pitfalls:** (1) skipping the edge-count check; (2) not confirming full connectivity.